In [1]:
import json, math, os, re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import jensenshannon

# Paths resolved relative to the notebook directory (assumes CWD = notebook dir)
_NB_DIR    = Path(os.path.abspath(''))
ROOT       = (_NB_DIR / "../../math-skills-llm/data/llm-label-semantic-space-grok").resolve()
REGRESSION = (_NB_DIR / "../../math-skills-llm/data/raw/regression").resolve()

from shape_metrics import effective_counts

# Paper models only
MODELS = [
    'nemotron-cascade-8b',
    'olmo-3-7b-think-rlvr',
    'qwen3-8b-thinking',
    'qwen3-32b',
]

MODEL_DISPLAY = {
    'nemotron-cascade-8b':  'Nemotron-Cascade-8B',
    'olmo-3-7b-think-rlvr': 'Olmo-3-7B-Think-RLVR',
    'qwen3-8b-thinking':    'Qwen3-8B',
    'qwen3-32b':            'Qwen3-32B',
}

FOLDER_PAT = re.compile(r'^(?P<model>.+?)_math_perturb_(?P<cond>original|hard|simple)_result$')
PARENT_PAT = re.compile(r'(H\d+)[a-z]$')

In [2]:
def get_parent(tag):
    if not isinstance(tag, str):
        return None
    tag = tag.strip()
    if not tag:
        return None

    m = PARENT_PAT.match(tag)
    if m:
        return m.group(1)

    if re.fullmatch(r'H\d+', tag):
        return tag

    if tag.startswith('N'):
        return 'N'

    return None


def extract_hseq_and_transition_counter(chunks):
    # Extract content-unit-level heuristic sets and transition counter.
    #
    # Each adjacent pair of content units contributes total mass 1.
    # If a unit has multiple heuristic parent tags, its transition mass is split uniformly
    # across all pairwise heuristic transitions, preserving content-unit boundaries.
    h_units = []
    h_freq = Counter()
    trans = Counter()

    for chunk in chunks:
        tags = chunk.get('ontology_tag', [])
        if isinstance(tags, str):
            tags = [tags]
        if not isinstance(tags, list):
            tags = []

        hs = sorted({
            p for p in (get_parent(t) for t in set(tags))
            if p is not None and p != 'N'
        })
        h_units.append(hs)
        for h in hs:
            h_freq[h] += 1.0

    for a, b in zip(h_units[:-1], h_units[1:]):
        if not a or not b:
            continue
        w = 1.0 / (len(a) * len(b))
        for ha in a:
            for hb in b:
                trans[(ha, hb)] += w

    return h_units, h_freq, trans


def load_chunks_from_file(path):
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        if not data:
            return []
        if isinstance(data[0], dict) and 'ontology_tag' in data[0]:
            return data
        flat = []
        for item in data:
            if isinstance(item, list):
                flat.extend([x for x in item if isinstance(x, dict)])
            elif isinstance(item, dict):
                flat.append(item)
        return [x for x in flat if 'ontology_tag' in x]

    if isinstance(data, dict):
        if isinstance(data.get('chunks'), list):
            return data['chunks']
        if isinstance(data.get('content_units'), list):
            return data['content_units']

    return []


def per_problem_stats(model, cond, base_dir=ROOT):
    folder = base_dir / f'{model}_math_perturb_{cond}_result'
    if not folder.exists():
        raise FileNotFoundError(f'Missing folder: {folder}')

    out = {}
    for fp in sorted(folder.glob('*.json')):
        problem_id = fp.stem
        chunks = load_chunks_from_file(fp)
        if not chunks:
            continue
        _, h_freq, trans = extract_hseq_and_transition_counter(chunks)
        out[problem_id] = {
            'n_units': len(chunks),
            'h_freq': h_freq,
            'trans': trans,
            'chunks': chunks,
        }
    return out


def jsd_between_counters(c1, c2):
    keys = sorted(set(c1.keys()) | set(c2.keys()))
    if not keys:
        return np.nan

    v1 = np.array([c1.get(k, 0.0) for k in keys], dtype=float)
    v2 = np.array([c2.get(k, 0.0) for k in keys], dtype=float)

    s1, s2 = v1.sum(), v2.sum()
    if s1 <= 0 or s2 <= 0:
        return np.nan

    p = v1 / s1
    q = v2 / s2
    return float(jensenshannon(p, q, base=2.0))

In [3]:
# Build per-problem distributions for each model/condition
stats_by_model = {}
for model in MODELS:
    stats_by_model[model] = {}
    for cond in ['original', 'simple', 'hard']:
        stats_by_model[model][cond] = per_problem_stats(model, cond, ROOT)

coverage_rows = []
for model in MODELS:
    o_ids = set(stats_by_model[model]['original'].keys())
    s_ids = set(stats_by_model[model]['simple'].keys())
    h_ids = set(stats_by_model[model]['hard'].keys())
    coverage_rows.append({
        'model': model,
        'n_original': len(o_ids),
        'n_simple': len(s_ids),
        'n_hard': len(h_ids),
        'n_common_original_simple': len(o_ids & s_ids),
        'n_common_original_hard': len(o_ids & h_ids),
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df


,model,n_original,n_simple,n_hard,n_common_original_simple,n_common_original_hard
0,nemotron-cascade-8b,115,115,115,115,115
1,olmo-3-7b-think-rlvr,115,115,115,115,115
2,qwen3-8b-thinking,115,115,115,115,115
3,qwen3-32b,114,115,114,114,113


In [4]:
# Per-problem D_JS^freq (heuristic frequency), ΔN_space_eff, and Δρ for (O→S) and (O→H) pairs.
per_problem_rows = []
summary_rows = []

for model in MODELS:
    model_stats = stats_by_model[model]

    for other in ['simple', 'hard']:
        common_ids = sorted(set(model_stats['original'].keys()) & set(model_stats[other].keys()))

        hf_vals, td_diff_vals = [], []

        for pid in common_ids:
            o = model_stats['original'][pid]
            t = model_stats[other][pid]

            hf_jsd = jsd_between_counters(o['h_freq'], t['h_freq'])

            o_space_eff, o_trans_eff = effective_counts(o['chunks'])
            t_space_eff, t_trans_eff = effective_counts(t['chunks'])

            o_rho = (o_trans_eff / o_space_eff) if (np.isfinite(o_space_eff) and o_space_eff > 0) else np.nan
            t_rho = (t_trans_eff / t_space_eff) if (np.isfinite(t_space_eff) and t_space_eff > 0) else np.nan
            td_diff = (t_rho - o_rho) if (np.isfinite(o_rho) and np.isfinite(t_rho)) else np.nan

            hf_vals.append(hf_jsd)
            td_diff_vals.append(td_diff)

            per_problem_rows.append({
                'model':                    model,
                'problem_id':               pid,
                'pair':                     f'original_vs_{other}',
                'heuristic_frequency_js_distance': hf_jsd,
                'original_N_space_eff':     o_space_eff,
                f'{other}_N_space_eff':     t_space_eff,
                'original_N_trans_eff':     o_trans_eff,
                f'{other}_N_trans_eff':     t_trans_eff,
                'transition_density_diff':  td_diff,
            })

        summary_rows.append({
            'model':  model,
            'pair':   f'original_vs_{other}',
            'n_common_problems': len(common_ids),
            'heuristic_frequency_js_distance_mean': float(np.nanmean(hf_vals))  if hf_vals else np.nan,
            'transition_density_diff_mean':          float(np.nanmean(td_diff_vals)) if td_diff_vals else np.nan,
        })

per_problem_js_distance_df = (pd.DataFrame(per_problem_rows)
                               .sort_values(['model', 'pair', 'problem_id'])
                               .reset_index(drop=True))
summary_js_distance_df = (pd.DataFrame(summary_rows)
                           .sort_values(['model', 'pair'])
                           .reset_index(drop=True))
summary_js_distance_df

,model,pair,n_common_problems,heuristic_frequency_js_distance_mean,transition_density_diff_mean
0,nemotron-cascade-8b,original_vs_hard,115,0.252936,0.104415
1,nemotron-cascade-8b,original_vs_simple,115,0.206211,-0.008798
2,olmo-3-7b-think-rlvr,original_vs_hard,115,0.242053,0.107245
3,olmo-3-7b-think-rlvr,original_vs_simple,115,0.190763,-0.052644
4,qwen3-32b,original_vs_hard,113,0.247964,0.032107
5,qwen3-32b,original_vs_simple,114,0.203808,-0.064378
6,qwen3-8b-thinking,original_vs_hard,115,0.241911,0.080466
7,qwen3-8b-thinking,original_vs_simple,115,0.193983,-0.025031


In [5]:
# Save per-problem metrics
out_dir = ROOT / 'js_divergence_results'
out_dir.mkdir(parents=True, exist_ok=True)

per_problem_js_distance_df.to_csv(out_dir / 'hard_perturb_per_problem_jsd.csv', index=False)
summary_js_distance_df.to_csv(out_dir / 'hard_perturb_per_problem_jsd_summary.csv', index=False)
print("Saved to:", out_dir)

Saved to: /data3/jongsong/math-skills-llm/data/llm-label-semantic-space-grok/js_divergence_results


In [6]:
# Load Pass@1 per model and condition from regression result folders
def load_pass_at_1(model: str, cond: str, regression_dir: Path = REGRESSION) -> float | None:
    folder = regression_dir / f'{model}_math_perturb_{cond}_result'
    if not folder.exists():
        return None
    correct, total = 0, 0
    for fp in sorted(folder.glob('*.json')):
        try:
            with fp.open('r') as f:
                data = json.load(f)
            if isinstance(data, list) and data:
                data = data[0]
            if not isinstance(data, dict):
                continue
            for key in ('correct', 'is_correct', 'accuracy'):
                val = data.get(key)
                if val is not None:
                    correct += int(bool(val))
                    total += 1
                    break
        except Exception:
            continue
    return correct / total if total > 0 else None

pass_at_1 = {m: {c: load_pass_at_1(m, c) for c in ['original', 'simple', 'hard']} for m in MODELS}

pass_at_1_df = pd.DataFrame([
    {'model': m, 'condition': c, 'pass_at_1': v}
    for m, conds in pass_at_1.items()
    for c, v in conds.items()
])
print(pass_at_1_df.pivot(index='model', columns='condition', values='pass_at_1').round(4))

condition             hard original simple
model                                     
nemotron-cascade-8b   None     None   None
olmo-3-7b-think-rlvr  None     None   None
qwen3-32b             None     None   None
qwen3-8b-thinking     None     None   None


In [7]:
# Final compact table: model-wise O->S / O->H metrics + one-sided tests
# Metrics shown per pair: heuristic_frequency_js_distance, n_space_eff_diff, transition_density_diff
# One-sided tests are run on paired problems with H1: (O->H) > (O->S).

pp = per_problem_js_distance_df.copy()

# Ensure n_space_eff_diff exists in the per-problem table.
if 'n_space_eff_diff' not in pp.columns:
    pp['n_space_eff_diff'] = np.where(
        pp['pair'] == 'original_vs_simple',
        pp['simple_N_space_eff'] - pp['original_N_space_eff'],
        np.where(
            pp['pair'] == 'original_vs_hard',
            pp['hard_N_space_eff'] - pp['original_N_space_eff'],
            np.nan,
        )
    )

metrics = ['heuristic_frequency_js_distance', 'n_space_eff_diff', 'transition_density_diff']

# 1) Pair-wise descriptive summary by model
summary_rows = []
for mdl, sub_m in pp.groupby('model', observed=False):
    for pair, sub_p in sub_m.groupby('pair', observed=False):
        row = {
            'model': mdl,
            'pair': pair,
            'n': len(sub_p),
        }
        for m in metrics:
            vals = pd.to_numeric(sub_p[m], errors='coerce')
            row[f'{m}_mean'] = float(np.nanmean(vals)) if len(vals) else np.nan
            row[f'{m}_median'] = float(np.nanmedian(vals)) if len(vals) else np.nan
        summary_rows.append(row)

model_pair_summary_df = pd.DataFrame(summary_rows).sort_values(['model', 'pair']).reset_index(drop=True)

# 2) One-sided tests by model: compare O->H vs O->S on same problem
# Build paired wide table per metric
base = pp[['model', 'problem_id', 'pair'] + metrics].copy()
base['other'] = base['pair'].map({'original_vs_simple': 'simple', 'original_vs_hard': 'hard'})

paired = (
    base
    .pivot_table(index=['model', 'problem_id'], columns='other', values=metrics, aggfunc='first')
)
paired.columns = [f'{m}_{o}' for (m, o) in paired.columns]
paired = paired.reset_index()

def _onesided(a: pd.Series, b: pd.Series):
    # H1: a > b
    d = a - b
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return {
            'n_paired': 0, 'mean_hard_minus_simple': np.nan, 'median_hard_minus_simple': np.nan,
            'pos': 0, 'neg': 0, 'ties': 0,
            'wilcoxon_W': np.nan, 'wilcoxon_p_one_sided': np.nan, 'sign_p_one_sided': np.nan,
        }

    pos = int((d > 0).sum())
    neg = int((d < 0).sum())
    ties = int((d == 0).sum())
    nonzero_n = pos + neg

    if nonzero_n > 0:
        w_res = stats.wilcoxon(a, b, alternative='greater', zero_method='wilcox', method='auto')
        w_stat = float(w_res.statistic)
        w_p = float(w_res.pvalue)
        sign_p = float(stats.binomtest(pos, nonzero_n, p=0.5, alternative='greater').pvalue)
    else:
        w_stat = np.nan
        w_p = np.nan
        sign_p = np.nan

    return {
        'n_paired': len(d),
        'mean_hard_minus_simple': float(np.nanmean(d)),
        'median_hard_minus_simple': float(np.nanmedian(d)),
        'pos': pos,
        'neg': neg,
        'ties': ties,
        'wilcoxon_W': w_stat,
        'wilcoxon_p_one_sided': w_p,
        'sign_p_one_sided': sign_p,
    }


# Overall + model-wise test rows
test_rows = []
for scope, sub in [('overall', paired)] + [(m, g) for m, g in paired.groupby('model', observed=False)]:
    for m in metrics:
        hs = f'{m}_hard'
        ss = f'{m}_simple'
        sub2 = sub.dropna(subset=[hs, ss]).copy()
        if len(sub2) == 0:
            out = {
                'scope': scope,
                'metric': m,
                'n_paired': 0,
                'mean_hard_minus_simple': np.nan,
                'median_hard_minus_simple': np.nan,
                'pos': 0,
                'neg': 0,
                'ties': 0,
                'wilcoxon_W': np.nan,
                'wilcoxon_p_one_sided': np.nan,
                'sign_p_one_sided': np.nan,
            }
        else:
            out = _onesided(sub2[hs], sub2[ss])
            out.update({'scope': scope, 'metric': m})
        test_rows.append(out)

one_sided_compact_df = pd.DataFrame(test_rows).sort_values(['scope', 'metric']).reset_index(drop=True)

# Display only requested outputs
print('=== Model-wise O->S / O->H Metrics ===')
display(model_pair_summary_df)

print('=== One-sided Test Results (H1: O->H > O->S) ===')
display(one_sided_compact_df)


=== Model-wise O->S / O->H Metrics ===


,model,pair,n,heuristic_frequency_js_distance_mean,heuristic_frequency_js_distance_median,n_space_eff_diff_mean,n_space_eff_diff_median,transition_density_diff_mean,transition_density_diff_median
0,nemotron-cascade-8b,original_vs_hard,115,0.252936,0.243919,0.069334,0.0,0.104415,0.0
1,nemotron-cascade-8b,original_vs_simple,115,0.206211,0.190032,-0.022844,0.0,-0.008798,0.0
2,olmo-3-7b-think-rlvr,original_vs_hard,115,0.242053,0.240482,0.082881,0.0,0.107245,0.0
3,olmo-3-7b-think-rlvr,original_vs_simple,115,0.190763,0.184411,-0.058476,0.0,-0.052644,0.0
4,qwen3-32b,original_vs_hard,113,0.247964,0.244825,0.067016,0.0,0.032107,0.0
5,qwen3-32b,original_vs_simple,114,0.203808,0.192354,-0.004086,0.0,-0.064378,0.0
6,qwen3-8b-thinking,original_vs_hard,115,0.241911,0.228551,0.139263,0.0,0.080466,0.0
7,qwen3-8b-thinking,original_vs_simple,115,0.193983,0.188617,-0.019682,0.0,-0.025031,0.0


=== One-sided Test Results (H1: O->H > O->S) ===


,n_paired,mean_hard_minus_simple,median_hard_minus_simple,pos,neg,ties,wilcoxon_W,wilcoxon_p_one_sided,sign_p_one_sided,scope,metric
0,115,0.046726,0.039003,80,35,0,5192.0,1.095056e-07,1.642221e-05,nemotron-cascade-8b,heuristic_frequency_js_distance
1,115,0.092178,0.000000,43,25,47,1540.0,1.246438e-02,1.923003e-02,nemotron-cascade-8b,n_space_eff_diff
2,115,0.113213,0.000000,42,26,47,1536.0,1.327543e-02,3.405934e-02,nemotron-cascade-8b,transition_density_diff
3,115,0.051290,0.049987,85,30,0,5178.0,1.349328e-07,1.430328e-07,olmo-3-7b-think-rlvr,heuristic_frequency_js_distance
4,115,0.141357,0.000000,50,26,39,1897.0,1.232094e-02,3.952590e-03,olmo-3-7b-think-rlvr,n_space_eff_diff
5,115,0.159889,0.000000,49,27,39,1999.0,2.759568e-03,7.720131e-03,olmo-3-7b-think-rlvr,transition_density_diff
6,458,0.047657,0.039767,326,132,0,81414.0,1.186570e-24,2.609515e-20,overall,heuristic_frequency_js_distance
7,458,0.116100,0.000000,196,101,161,29305.0,6.294807e-07,1.897744e-08,overall,n_space_eff_diff
8,458,0.119009,0.000000,187,111,160,29134.5,2.040786e-06,6.302132e-06,overall,transition_density_diff
9,113,0.044631,0.035601,79,34,0,4927.0,5.071156e-07,1.386557e-05,qwen3-32b,heuristic_frequency_js_distance


In [8]:
# Paper table: CoT structural analysis under perturbation
# Columns: Pass@1 (O/S/H) | D_JS^freq (O→S, O→H) | ΔN_space_eff (O→S, O→H) | Δρ (O→S, O→H)
# * = one-sided Wilcoxon p < 0.05 for H1: (O→H) > (O→S), tested per model

SIG_THRESHOLD = 0.05

# Build significance lookup: (model_name, metric_key) -> bool
sig_lookup = {
    (row['scope'], row['metric']): row['wilcoxon_p_one_sided'] < SIG_THRESHOLD
    for _, row in one_sided_compact_df.iterrows()
    if row['scope'] != 'overall'
}

def _fmt(val, dec=3):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '—'
    return f'{val:.{dec}f}'

def _fmt_pct(val, dec=1):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '—'
    return f'{val * 100:.{dec}f}'

def _get(pair, col):
    mask = (model_pair_summary_df['model'] == model) & (model_pair_summary_df['pair'] == pair)
    sub = model_pair_summary_df.loc[mask, col]
    return float(sub.iloc[0]) if len(sub) else np.nan

def _sig(metric):
    return '*' if sig_lookup.get((model, metric), False) else ''

table_rows = []
for model in MODELS:
    table_rows.append({
        'Model':           MODEL_DISPLAY[model],
        'Pass@1 O':        _fmt_pct(pass_at_1[model]['original']),
        'Pass@1 S':        _fmt_pct(pass_at_1[model]['simple']),
        'Pass@1 H':        _fmt_pct(pass_at_1[model]['hard']),
        'D_JS (O→S)':      _fmt(_get('original_vs_simple', 'heuristic_frequency_js_distance_mean')),
        'D_JS (O→H)':      _fmt(_get('original_vs_hard',   'heuristic_frequency_js_distance_mean'))
                           + _sig('heuristic_frequency_js_distance'),
        'ΔN_sp (O→S)':     _fmt(_get('original_vs_simple', 'n_space_eff_diff_mean')),
        'ΔN_sp (O→H)':     _fmt(_get('original_vs_hard',   'n_space_eff_diff_mean'))
                           + _sig('n_space_eff_diff'),
        'Δρ (O→S)':        _fmt(_get('original_vs_simple', 'transition_density_diff_mean')),
        'Δρ (O→H)':        _fmt(_get('original_vs_hard',   'transition_density_diff_mean'))
                           + _sig('transition_density_diff'),
    })

paper_table_df = pd.DataFrame(table_rows).set_index('Model')
display(paper_table_df)
print('\n* = one-sided Wilcoxon p < 0.05 (H1: O→H > O→S, per model)')

,Pass@1 O,Pass@1 S,Pass@1 H,D_JS (O→S),D_JS (O→H),ΔN_sp (O→S),ΔN_sp (O→H),Δρ (O→S),Δρ (O→H)
Model,,,,,,,,,
Nemotron-Cascade-8B,—,—,—,0.206,0.253*,-0.023,0.069*,-0.009,0.104*
Olmo-3-7B-Think-RLVR,—,—,—,0.191,0.242*,-0.058,0.083*,-0.053,0.107*
Qwen3-8B,—,—,—,0.194,0.242*,-0.020,0.139*,-0.025,0.080*
Qwen3-32B,—,—,—,0.204,0.248*,-0.004,0.067*,-0.064,0.032*



* = one-sided Wilcoxon p < 0.05 (H1: O→H > O→S, per model)
